In [1]:
%matplotlib inline
import os
import sys
from os.path import join as pjoin
from tifffile import imread, imwrite, TiffFile
import numpy as np
import shutil
import matplotlib.pyplot as plt
from glob import glob
import pandas as pd
import cv2
from tqdm import tqdm
import subprocess
from scipy.ndimage import gaussian_filter,median_filter
from scipy.interpolate import interp1d
from scipy.signal import detrend, butter, filtfilt

project_root = '/home/lsh/WF_GoNogo'
if project_root not in sys.path:
    sys.path.append(project_root)
from utils.wfield_utils import *

In [2]:
config_path = "/home/lsh/WF_GoNogo/config/A095_config.yaml"
def analyze_mice(config_path):
    config = load_config(config_path)
    for session in config["sessions"]:
        analyze_session(config, session)
    print(f"all done")

In [ ]:
# functions
import yaml
def load_config(config_path):
    """load the YAML config"""
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
    return config

# analyze_mice(config_path)
def analyze_session(config,session):
    print(f"Analyzing session: {session}")
    # path 
    processPath = config["paths"]["process"].format(
        base_dir=config["base_dir"], mouse_id=config["mouse_id"], session=session
    )
    os.makedirs(processPath, exist_ok=True)
    timePath = config["paths"]["time"].format(
        base_dir=config["base_dir"], mouse_id=config["mouse_id"], session=session
    )
    os.makedirs(timePath, exist_ok=True)

    U_neuro = np.load(pjoin(processPath, "U_blue.npy"))
    U_hemo = np.load(pjoin(processPath, "U_violet.npy"))
    V_neuro = np.load(pjoin(processPath, "V_blue.npy"))
    V_hemo = np.load(pjoin(processPath, "V_violet.npy"))
    t_neuro = np.load(pjoin(timePath),"widefield_timestamps_blue.npy")
    t_hemo = np.load(pjoin(timePath),"widefield_timestamps_violet.npy")
    
    V_neuro_hemocorr, hemocorr_t = hemo_correct(U_neuro, V_neuro, t_neuro, U_hemo, V_hemo, t_hemo)

    np.save(pjoin(processPath, "V_neuro_hemocorr.npy"), V_neuro_hemocorr)
    np.save(pjoin(timePath, "hemocorr_t.npy"), hemocorr_t)

    ## STA, stimulus triggered average
    stim_timestamps = np.load(pjoin(timePath, "stim_timestamp.npy"))
    trial_timestamps = np.load(pjoin(timePath, "trial_timestamp.npy"))
    sta, t_window = compute_sta(stim_timestamps, trial_timestamps, V_neuro_hemocorr, hemocorr_t, pre_stim=0.5, post_stim=1.0, wf_framerate=20)
    np.save(pjoin(processPath, "sta_neuro_hemocorr.npy"), sta)
    np.save(pjoin(processPath, "sta_time_window.npy"), t_window)
        # trial type (go/nogo stimuli)

    ## trial type response average
    # read trial type .txt file
    trial_types = 
    
    
    






SyntaxError: incomplete input (2167184293.py, line 10)

In [8]:
import numpy as np
from scipy import signal
from scipy.interpolate import interp1d
from skimage.transform import resize
import matplotlib.pyplot as plt


def hemo_correct(U_neuro, V_neuro, t_neuro, U_hemo, V_hemo, t_hemo):
    """
    Remove hemodynamic signals from widefield
    
    Parameters:
    -----------
    U_neuro, V_neuro : ndarray
        Widefield SVD components for neural signal (blue light)
    t_neuro : ndarray
        Timestamps for neural signal
    U_hemo, V_hemo : ndarray
        Widefield SVD components for hemodynamic signal (violet light)
    t_hemo : ndarray
        Timestamps for hemodynamic signal
    
    Returns:
    --------
    V_neuro_hemocorr : ndarray
        Neural V with hemodynamic component subtracted
    hemocorr_t : ndarray
        Timestamps (same as t_hemo)
    
    Notes:
    ------
    Gets scaling factor for hemo signal onto neuro signal, converts hemo V's
    into neuro U-space, baseline-subtracts and scales hemo signal, subtracts
    neuro-estimated hemo signal from neuro signal.
    
    Function based off cortexlab/widefield/hemo_correct_local (Kenneth)
    """
    
    ## Set parameters
    
    # Spatial downsample factor
    # (pixel traces reconstructed at this scale, 5 seems fine)
    px_downsample = 5
    
    # Frequency for filtering heartbeat
    heartbeat_freq = [8, 12]
    
    ## Overlap neuro/hemo signals
    
    # Change hemo V from hemo to neuro U basis
    U_neuro_flat = U_neuro.reshape(-1, U_neuro.shape[2])
    U_hemo_flat = U_hemo.reshape(-1, U_hemo.shape[2])
    
    V_hemo_Un = U_neuro_flat.T @ U_hemo_flat @ V_hemo
    
    # Interpolate neuro to hemo timepoints (captured alternating)
    # (shifting neuro to hemo works much better than opposite)
    interp_func = interp1d(t_neuro, V_neuro, kind='linear', 
                           axis=1, fill_value='extrapolate')
    V_neuro_th = interp_func(t_hemo)
    
    # Set the timestamps
    hemocorr_t = t_hemo
    
    ## Get scale factor to match hemo onto neuro
    ## (using the heartbeat frequency)
    
    # Set frames to use (middle percentile: avoid edge artifacts)
    n_frames = V_neuro.shape[1]
    use_frame_range = np.percentile(np.arange(n_frames), [12.5, 87.5]).astype(int)
    use_frames = np.arange(use_frame_range[0], use_frame_range[1] + 1)
    
    # Filter V's at heartbeat frequency
    wf_framerate = 1 / np.mean(np.diff(t_neuro))
    b, a = signal.butter(2, np.array(heartbeat_freq) / (wf_framerate / 2), 
                         btype='bandpass')
    
    V_neuro_th_heartbeat = signal.filtfilt(b, a, V_neuro_th, axis=1)
    V_hemo_Un_heartbeat = signal.filtfilt(b, a, V_hemo_Un, axis=1)
    
    # Downsample U
    # Note: Using order=0 for nearest neighbor interpolation
    U_neuro_downsamp = resize(U_neuro, 
                             (U_neuro.shape[0] // px_downsample,
                              U_neuro.shape[1] // px_downsample,
                              U_neuro.shape[2]),
                             order=0, preserve_range=True, anti_aliasing=False)
    
    # Get pixel traces (t x px) from downsampled U
    px_neuro_heartbeat = svd2px(U_neuro_downsamp, 
                                V_neuro_th_heartbeat[:, use_frames])
    px_neuro_heartbeat = px_neuro_heartbeat.reshape(-1, len(use_frames)).T
    
    px_hemo_heartbeat = svd2px(U_neuro_downsamp, 
                               V_hemo_Un_heartbeat[:, use_frames])
    px_hemo_heartbeat = px_hemo_heartbeat.reshape(-1, len(use_frames)).T
    
    # Get scaling of violet to blue from heartbeat
    # scaling = cov(neuro-mean,hemo-mean)/var(hemo-mean)
    px_neuro_centered = px_neuro_heartbeat - px_neuro_heartbeat.mean(axis=0)
    px_hemo_centered = px_hemo_heartbeat - px_hemo_heartbeat.mean(axis=0)
    
    hemo_scale_px_downsamp = (np.sum(px_neuro_centered * px_hemo_centered, axis=0) / 
                              np.sum(px_hemo_centered ** 2, axis=0))
    
    # Get transform matrix to convert scaling from pixel-space to V-space
    U_neuro_downsamp_flat = U_neuro_downsamp.reshape(-1, U_neuro_downsamp.shape[2])
    hemo_scale_V_tform = (np.linalg.pinv(U_neuro_downsamp_flat) @ 
                         np.diag(hemo_scale_px_downsamp) @ 
                         U_neuro_downsamp_flat)
    
    ## Hemo-correct neuro signal
    
    # Using scaled detrended hemo signal
    V_hemo_Un_detrend = signal.detrend(V_hemo_Un, axis=1)
    neuro_hemo_estimation = (V_hemo_Un_detrend.T @ hemo_scale_V_tform.T).T
    V_neuro_hemocorr = V_neuro_th - neuro_hemo_estimation
    
    ## Check results (optional)
    check_results = False
    
    if check_results:
        # Image the V hemo scale
        plt.figure(figsize=(10, 8))
        hemo_scale_img = hemo_scale_px_downsamp.reshape(
            U_neuro_downsamp.shape[0], U_neuro_downsamp.shape[1])
        vmax = np.max(np.abs([hemo_scale_img.min(), hemo_scale_img.max()]))
        plt.imshow(hemo_scale_img, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        plt.colorbar()
        plt.title('Hemo to neuro scale factor')
        plt.axis('equal')
        
        # Note: ROI selection and spectrum plotting would require additional
        # helper functions (AP_svd_roi) that aren't included here
        
        plt.show()
    
    return V_neuro_hemocorr, hemocorr_t


def svd2px(U, V):
    """
    Reconstruct pixel space from SVD components
    
    Parameters:
    -----------
    U : ndarray (height, width, n_components)
        Spatial components
    V : ndarray (n_components, n_timepoints)
        Temporal components
    
    Returns:
    --------
    px : ndarray (height, width, n_timepoints)
        Reconstructed pixel data
    """
    h, w, n_comp = U.shape
    U_flat = U.reshape(-1, n_comp)
    px_flat = U_flat @ V
    px = px_flat.reshape(h, w, -1)
    return px


# # Example usage:
# if __name__ == "__main__":
#     # This is example code showing how to use the function
#     # You would need to load your actual data
    
#     # Example with random data (replace with actual data loading)
#     # U_neuro = np.random.randn(100, 100, 50)  # spatial components
#     # V_neuro = np.random.randn(50, 1000)      # temporal components
#     # t_neuro = np.linspace(0, 100, 1000)      # timestamps
#     # 
#     # U_hemo = np.random.randn(100, 100, 50)
#     # V_hemo = np.random.randn(50, 1000)
#     # t_hemo = np.linspace(0, 100, 1000)
#     # 
#     # V_corrected, t_corrected = hemo_correct(
#     #     U_neuro, V_neuro, t_neuro,
#     #     U_hemo, V_hemo, t_hemo
#     # )
    
#     print("Function defined. Load your data and call hemo_correct() to use.")

In [ ]:
def compute_STA(stim_timestamps, trial_timestamps, V_neuro_hemocorr, hemocorr_t, pre_stim=0.5, post_stim=1.0, wf_framerate=20):

    """Compute stimulus-triggered average (STA) for widefield data."""

    n_pre = int(pre_stim * wf_framerate)
    n_post = int(post_stim * wf_framerate)
    n_window = n_pre + n_post + 1
    t_window = np.linspace(-pre_stim, post_stim, n_window)

    n_pixels, n_frames = V_neuro_hemocorr.shape
    sta_accum = np.zeros((n_pixels, n_window))
    count = 0

    for stim_t in stim_timestamps:
       
        idx = np.argmin(np.abs(hemocorr_t - stim_t))
        
        if idx - n_pre < 0 or idx + n_post >= n_frames:
            continue

        snippet = V_neuro_hemocorr[:, idx - n_pre : idx + n_post + 1]
        sta_accum += snippet
        count += 1


    if count > 0:
        sta = sta_accum / count
    else:
        print("Warning: No valid stimuli found within time window.")
        sta = np.zeros_like(sta_accum)

    return sta, t_window

def compute_trial_type_response(stim_timestamps, trial_timestamps, V_neuro_hemocorr, hemocorr_t, wf_framerate=20):
    """Compute average response for different trial types (go/nogo)."""

    # 1 Hit 2 Miss 3 CR 4 FA
    trial_type_dict = {1: 'Hit', 2: 'Miss', 3: 'CR', 4: 'FA'}

